# Stage 12 — Paper trading journal (debit spread + option buying, forward-only)

Neither strategy has been traded with real money yet. Both already have a fully
backtested trade lifecycle (Stages 9/11) on real historical data since 2023 —
what hasn't been checked is whether that same signal keeps producing a similar
outcome distribution **going forward**, on trades nobody hand-picked with
hindsight.

**No new entry/exit rule is added here.** This refreshes the exact same
shortlist notebooks 16/19 already refresh, using the same
`process_instrument_incremental()` + trade-lifecycle recompute those notebooks
use — then keeps every trade whose `entry_date` falls on or after this
journal's start date (fixed the first time this ran, saved to
`config/paper_trading_start.json`, never moved afterward).

**Still entirely paper.** Nothing here places, closes, or modifies any real
order — this only journals what the signal would have done, so the results can
be reviewed after a month or two (win rate, avg R, total R — same metrics the
historical backtests already use) before risking real capital.

Run this once a day — cheap, incremental fetch only. Re-running more often just
refreshes OPEN trades' current unrealized R, no harm.

In [1]:
import sys
sys.path.append("..")

import pandas as pd

ranking_debit = pd.read_excel("../exports/RANKING_debit_spread.xlsx")
ranking_option = pd.read_excel("../exports/RANKING_option_buying.xlsx")

# Same shortlist rule notebooks 16/19 already use — outlier-reliance-clean,
# top 20 by score, NIFTY/BANKNIFTY always included if clean. Not a new
# selection rule, just reused so the paper journal watches exactly what the
# live screeners already consider tradeable.
def shortlist(ranking):
    clean = ranking[ranking["outlier_reliance_flag"].isna() | (ranking["outlier_reliance_flag"] == "")]
    top_20 = clean.head(20)
    must_include = ranking[ranking["symbol"].isin(["NIFTY", "BANKNIFTY"])]
    must_include_clean = must_include[must_include["outlier_reliance_flag"].isna() | (must_include["outlier_reliance_flag"] == "")]
    return pd.concat([top_20, must_include_clean]).drop_duplicates(subset="symbol").reset_index(drop=True)

debit_candidates = shortlist(ranking_debit)
option_candidates = shortlist(ranking_option)

print(f"Debit spread shortlist: {len(debit_candidates)} symbols")
print(f"Option buying shortlist: {len(option_candidates)} symbols")

Debit spread shortlist: 20 symbols
Option buying shortlist: 21 symbols


In [2]:
from datetime import datetime
from src.auth import get_kite
from src.instruments import build_nse_token_lookup, get_index_token
from src.debit_spread_scale_pipeline import process_instrument_incremental as refresh_debit_spread
from src.option_buying_scale_pipeline import process_instrument_incremental as refresh_option_buying

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)
index_symbols = {"NIFTY": ("NIFTY 50", "NSE"), "BANKNIFTY": ("NIFTY BANK", "NSE")}

def token_for(symbol):
    if symbol in index_symbols:
        tradingsymbol, exchange = index_symbols[symbol]
        return get_index_token(kite, tradingsymbol, exchange)
    return token_lookup.get(symbol)

def refresh(symbols, refresh_fn, label):
    for i, symbol in enumerate(symbols, start=1):
        try:
            token = token_for(symbol)
            if token is None:
                print(f"[{label} {i}/{len(symbols)}] {symbol} — no token found, skipping")
                continue
            refresh_fn(symbol, token, kite, to_date=datetime.now())
            print(f"[{label} {i}/{len(symbols)}] {symbol} — refreshed")
        except Exception as e:
            print(f"[{label} {i}/{len(symbols)}] {symbol} — FAILED: {e}")

refresh(debit_candidates["symbol"].tolist(), refresh_debit_spread, "debit spread")
refresh(option_candidates["symbol"].tolist(), refresh_option_buying, "option buying")

Fetching 2026-06-30 to 2026-10-28...
Saved 6529 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/POWERINDIA_60minute.csv
Saved 69 trades to /Users/hgmac/Documents/3_trading/Quant/exports/debit_spread/POWERINDIA_trades.csv
[debit spread 1/20] POWERINDIA — refreshed
Fetching 2026-06-30 to 2026-10-28...
Saved 6589 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/NIFTY_60minute.csv
Saved 64 trades to /Users/hgmac/Documents/3_trading/Quant/exports/debit_spread/NIFTY_trades.csv
[debit spread 2/20] NIFTY — refreshed
Fetching 2026-06-30 to 2026-10-28...
Saved 3315 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/SWIGGY_60minute.csv
Saved 31 trades to /Users/hgmac/Documents/3_trading/Quant/exports/debit_spread/SWIGGY_trades.csv
[debit spread 3/20] SWIGGY — refreshed
Fetching 2026-06-30 to 2026-10-28...
Saved 6529 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/CDSL_60minute.csv
Saved 71 trades to /Users/hgmac/Documents/3_trading/Quant/exports/debit_sprea

## The journal — forward trades only, since tracking started

In [3]:
from src.paper_trading import build_journal, summarize, save_journal, get_start_date

debit_journal = build_journal(debit_candidates["symbol"].tolist(), "debit_spread")
option_journal = build_journal(option_candidates["symbol"].tolist(), "option_buying")

journal = pd.concat([debit_journal, option_journal], ignore_index=True)

print(f"Debit spread paper-tracking since {get_start_date('debit_spread').date()}")
print(f"Option buying paper-tracking since {get_start_date('option_buying').date()}")

if journal.empty:
    print("\nNo forward trades yet — nothing has fired since tracking started. Re-run daily.")
else:
    path = save_journal(journal)
    print(f"\n{len(journal)} forward trade(s) so far, saved to {path}")

Debit spread paper-tracking since 2026-09-07
Option buying paper-tracking since 2026-09-07

66 forward trade(s) so far, saved to /Users/hgmac/Documents/3_trading/Quant/logs/paper_trading_journal.csv


In [4]:
if not journal.empty:
    summary = summarize(journal)
    print("SUMMARY — by strategy and status (OPEN = live unrealized R, CLOSED = resolved):")
    display(summary)

    print("\nFull journal, most recent entry first:")
    display(journal.sort_values("entry_date", ascending=False))

SUMMARY — by strategy and status (OPEN = live unrealized R, CLOSED = resolved):


,strategy,status,n,win_rate_pct,avg_r,median_r,total_r
0,debit_spread,CLOSED,39,66.7,0.571,1.50,22.28
1,debit_spread,OPEN,1,100.0,0.400,0.40,0.40
2,option_buying,CLOSED,25,76.0,1.028,1.24,25.69
3,option_buying,OPEN,1,100.0,0.400,0.40,0.40



Full journal, most recent entry first:


,strategy,symbol,direction,status,entry_date,entry_price,stop_price,target_price,exit_date,exit_price,exit_reason,candles_held,pct_move,r_achieved,max_adverse_pct,initial_stop_price,final_stop_price,trailing_activated,four_h_direction
65,option_buying,KPITTECH,UP,OPEN,2026-10-28 13:15:00,544.15,NaN,NaN,2026-10-28 14:15:00,550.70,OPEN,1,1.20,0.40,0.11,527.80,527.80,False,UP
39,debit_spread,KPITTECH,UP,OPEN,2026-10-28 13:15:00,544.15,527.80,568.68,2026-10-28 14:15:00,550.70,OPEN,1,1.20,0.40,0.11,NaN,NaN,NaN,NaN
38,debit_spread,NIFTY,DOWN,CLOSED,2026-10-28 10:15:00,22585.35,22755.70,22329.82,2026-10-28 12:15:00,22755.70,STOP,2,-0.75,-1.00,0.85,NaN,NaN,NaN,NaN
37,debit_spread,APOLLOHOSP,UP,CLOSED,2026-10-26 13:15:00,7782.50,7682.49,7932.52,2026-10-27 11:15:00,7932.52,TARGET,4,1.93,1.50,0.00,NaN,NaN,NaN,NaN
36,debit_spread,INFY,DOWN,CLOSED,2026-10-26 12:15:00,1143.25,1172.31,1099.66,2026-10-28 14:15:00,1116.25,TIME,14,2.36,0.93,1.97,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3,debit_spread,CDSL,DOWN,CLOSED,2026-09-09 10:15:00,1368.30,1387.45,1339.58,2026-09-11 09:15:00,1339.58,TARGET,11,2.10,1.50,0.74,NaN,NaN,NaN,NaN
2,debit_spread,SUZLON,UP,CLOSED,2026-09-09 10:15:00,46.03,45.53,46.78,2026-09-09 14:15:00,45.53,STOP,4,-1.09,-1.00,1.59,NaN,NaN,NaN,NaN
1,debit_spread,APOLLOHOSP,UP,CLOSED,2026-09-07 13:15:00,8797.50,8725.86,8904.97,2026-09-09 09:15:00,8904.97,TARGET,8,1.22,1.50,0.60,NaN,NaN,NaN,NaN
40,option_buying,GODREJPROP,DOWN,CLOSED,2026-09-07 11:15:00,1920.40,NaN,NaN,2026-09-09 13:15:00,1880.50,TIME,14,2.08,1.06,0.50,1958.15,1911.05,True,DOWN
